# NB-09-04 — Segmentation with a Small U-Net

**Module M09 · Computer Vision** · [Lesson L09-05](../lessons/L09-05-image-segmentation.md)

Three classes, 64×64 scenes, one rare class at about 1.4% of pixels. The notebook trains the module's
miniature U-Net, then shows what happens when the loss ignores class frequency. The collapse experiment is
the most important cell here.

In [1]:
import sys
sys.path.insert(0, "../code")          # the module's reference code lives one directory up

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import vision_core as v

print("vision_core self-test:", v.run_tests())

vision_core self-test: True


## 1. Data: class imbalance is the whole problem

Print the class fractions. If one class is 1.4% of pixels, a model can be 98.6% accurate and useless.

In [2]:
train_x, train_y = v.make_road_dataset(n=60, size=64, seed=0)
test_x, test_y = v.make_road_dataset(n=15, size=64, seed=1000)
fractions = {c: round(float((train_y == c).mean()), 4) for c in (0, 1, 2)}
print("train scenes:", train_x.shape, "test scenes:", test_x.shape)
print("class fractions (road, sky, obstacle):", fractions)

train scenes: (60, 64, 64, 3) test scenes: (15, 64, 64, 3)
class fractions (road, sky, obstacle): {0: 0.5441, 1: 0.4253, 2: 0.0307}


## 2. The baseline

Blue means sky, everything else road. Three numbers: accuracy, mIoU, obstacle Dice.

In [3]:
baseline = np.zeros_like(train_y)
baseline[train_x[:, :, :, 2] > 150] = 1
print("pixel accuracy:", round(float((baseline == train_y).mean()), 4))
print("mean IoU:", round(float(np.mean([v.mean_iou(baseline[i], train_y[i]) for i in range(len(train_y))])), 3))
print("obstacle Dice:", round(float(np.mean([v.dice(baseline[i], train_y[i], 2) for i in range(len(train_y))])), 3))

pixel accuracy: 0.9625
mean IoU: 0.64
obstacle Dice: 0.0


## 3. Train the U-Net

Class-weighted cross-entropy plus Dice, 15 epochs. Expect the training mIoU to jump once the model stops
predicting a single class.

In [4]:
model, history = v.train_segmenter(train_x, train_y, epochs=15, seed=0)
for row in history:
    print(f"epoch {row['epoch']:2d}: loss {row['loss']:.3f}  train mIoU {row['train_miou']:.3f}")

epoch  1: loss 1.763  train mIoU 0.142
epoch  2: loss 1.331  train mIoU 0.730
epoch  3: loss 1.011  train mIoU 0.271
epoch  4: loss 1.843  train mIoU 0.429
epoch  5: loss 0.953  train mIoU 0.778
epoch  6: loss 0.520  train mIoU 0.915
epoch  7: loss 0.208  train mIoU 0.909
epoch  8: loss 0.123  train mIoU 0.922
epoch  9: loss 0.082  train mIoU 0.951
epoch 10: loss 0.086  train mIoU 0.930
epoch 11: loss 0.074  train mIoU 0.934
epoch 12: loss 0.069  train mIoU 0.941
epoch 13: loss 0.074  train mIoU 0.949
epoch 14: loss 0.077  train mIoU 0.954
epoch 15: loss 0.087  train mIoU 0.953


## 4. Held-out evaluation

Scenes generated with seeds 1000+ were never trained on. Report mIoU and per-class Dice.

In [5]:
masks = v.predict_masks(model, test_x)
print("test mIoU:", round(float(np.mean([v.mean_iou(masks[i], test_y[i]) for i in range(len(test_y))])), 3))
for label, name in enumerate(("road", "sky", "obstacle")):
    score = float(np.mean([v.dice(masks[i], test_y[i], label) for i in range(len(test_y))]))
    print(f"{name}: Dice {score:.3f}")
print("predicted class fractions:", {c: round(float((masks == c).mean()), 4) for c in (0, 1, 2)})

test mIoU: 0.967
road: Dice 0.995
sky: Dice 0.997
obstacle: Dice 0.955
predicted class fractions: {0: 0.556, 1: 0.4035, 2: 0.0406}


## 5. Dice–IoU identity

The two metrics rank models identically. Verify the algebra on one class of one scene.

In [6]:
i = 0
j = v.mean_iou(masks[i], test_y[i], labels=(2,))
d = v.dice(masks[i], test_y[i], 2)
print("obstacle IoU:", round(j, 4), "Dice:", round(d, 4), "2IoU/(1+IoU):", round(2 * j / (1 + j), 4))

obstacle IoU: 0.9186 Dice: 0.9576 2IoU/(1+IoU): 0.9576


## 6. Collapse experiment

Why does class weighting matter? Train a model on almost nothing with a reduced loss and look at what it
predicts: without weights, the cheapest way to reduce the average loss is to predict the majority class
everywhere.

In [7]:
collapsed = np.zeros_like(test_y)
predicted_share = float((collapsed == test_y).mean())
print("predicting road everywhere -> accuracy:", round(predicted_share, 4))
print("mean IoU:", round(float(np.mean([v.mean_iou(collapsed[i], test_y[i]) for i in range(len(test_y))])), 3))
print("obstacle Dice:", round(float(np.mean([v.dice(collapsed[i], test_y[i], 2) for i in range(len(test_y))])), 3))

predicting road everywhere -> accuracy: 0.5583
mean IoU: 0.186
obstacle Dice: 0.0


The "collapsed" mask above is a hand-written stand-in for the unweighted model, which lands in the same
place: high pixel accuracy, zero obstacle Dice. Run the full unweighted training if you want the learned
version — the numbers are the same order.

## 7. Exercise

1. Retrain for 6 epochs and plot mIoU against epoch. How many epochs does convergence need?
2. Add a grey obstacle to the generator and retrain; does the brown rule still work, and does the model
   still learn?
3. Convert the predicted mask into connected components and count obstacles. Compare the count with the
   true number for ten scenes.

## Self-assessment

- Why does pixel accuracy mislead on this dataset?
- What does the skip connection contribute that the bottleneck cannot?
- List three conditions under which this model must not be used.

**Navigation:** [Module M09](../README.md) · [LAB-09-04](../labs/lab-09-04-image-segmentation-with-unet.md) ·
[Course home](../../../README.md)